# 05 — Writeup: does the pairs-trading edge survive?

**No — not for these pairs, at this frequency, at realistic cost.**

Three pairs were specified in advance from economic reasoning: **WM/RSG** (a duopoly in
North American waste collection), **FOXA/FOX** (two share classes of one company) and
**SPY/VOO** (two wrappers on the same index). They span the full range of how tightly two
assets can be linked — from an economic argument to a near-mechanical identity.

All three lose money after costs, in-sample and out-of-sample, six cells out of six. The
gross returns hover around zero; a transaction-cost drag of 0.6–1.1% a year, fully
accounted for by how often the strategy trades, decides every sign.

That is the finding. This notebook is about **why it should be believed** — which, for a
negative result, means showing that the machinery was capable of producing a positive one
and that the negative did not come from a bug. The five methodological principles below
are not a preamble to the result; each one is load-bearing, and §2.3 shows what the
headline would have been had one of them been dropped.

*Numbers below are read from the same `reports/results/metrics.csv` as
`04_backtest_results.ipynb`. Tables there, argument here.*


In [ ]:
import dataclasses
import json
from pathlib import Path

import pandas as pd

from pairs_teardown.config import load_config
from pairs_teardown.data.loaders import load_or_download
from pairs_teardown.study import run_pair

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

cfg = load_config(ROOT / "configs" / "pairs.yaml")
metrics = pd.read_csv(ROOT / "reports" / "results" / "metrics.csv")
manifest = json.loads((ROOT / "reports" / "results" / "run_manifest.json").read_text())
OFFICIAL = [p.name for p in cfg.official_pairs]

pd.set_option("display.precision", 3)


## 1. What was actually tested

For each pair, on log prices, over 2015-01-01 to 2024-12-31:

1. **Spread.** A rolling 60-day OLS hedge ratio builds the spread. Rolling is used here
   because static-hedge spreads fail an ADF stationarity test for WM/RSG (p ≈ 0.10) and
   FOXA/FOX (p ≈ 0.29) — the method's core assumption does not hold with a fixed ratio.
2. **Signal.** A 60-day rolling z-score of that spread. Enter at |z| ≥ 2.0, exit at
   |z| ≤ 0.5, hold the previous position in between (hysteresis, so the position does not
   churn while z hovers near a threshold).
3. **Sizing.** A **static** hedge ratio, fit by OLS **on in-sample data only** and then
   frozen — deliberately a different estimator from the signal's (see §2.1).
4. **Execution.** The position decided on day *t* is executed at day *t+1* prices.
5. **Costs.** 1 bp commission + 5 bps slippage per side, charged on `(1 + |g|)` of
   notional for every unit of position change.

Parameters were frozen before the out-of-sample period was scored, and it was scored
once. `configs/pairs.yaml` is committed, so the parameters are a record rather than a
claim:


In [ ]:
for k in ["window", "entry", "exit", "signal_hedge", "sizing_hedge",
          "commission_bps", "slippage_bps", "data_start", "in_sample_end", "data_end"]:
    print(f"{k:<16} {manifest[k]}")


## 2. The five principles

### 2.1 No look-ahead bias

**What it demands.** Nothing computed for day *t* may use information from after day *t*.

**What was done.** Three separate mechanisms, each with a test that fails if it is
removed:

| Mechanism | Guard |
|---|---|
| Positions lagged one bar before P&L | `test_engine.py` — altering a *future* price must not change any *past* P&L value |
| Sizing hedge ratio fit on in-sample data only | `test_study.py` — shocking out-of-sample prices must not move the fitted ratio, or any in-sample metric |
| Z-score uses a trailing window only | `test_spread.py` |

**What it cost.** The in-sample-only rule is not free. `03_backtest_explore.ipynb`, written
before the pipeline was assembled, fit the sizing hedge on the *full* sample — the natural
thing to do when exploring — and reports materially different numbers as a result. Its
figures are kept as diagnostics and explicitly labelled superseded, rather than quietly
deleted, because the gap between the two is itself the lesson.

**The near-miss worth recording.** An early version used the *rolling* hedge ratio for
position sizing as well as for the signal. It seemed like a consistency improvement. On
synthetic data with a known true ratio it returned **−61.6%** where the correct sizing
returned **+521%**: a noisy rolling estimate drifts toward zero, and a hedge ratio near
zero silently converts a market-neutral spread into an outright directional bet on one
leg. This is now blocked in three places — a runtime guard on the hedge series' standard
deviation, a hard rejection of `sizing_hedge: "rolling"` in config validation, and
`test_engine.py::test_correct_hedge_sign_neutralizes_common_move_wrong_sign_does_not`.

The general point: **a backtest that is wrong in this way looks better, not worse.** That
asymmetry is the entire argument for building the guards before trusting any number.


### 2.2 Survivorship bias — acknowledged, not engineered around

**The exposure.** All six tickers were chosen in 2025 from companies that still exist,
still trade, and still have a live counterpart. WM, RSG, KO, PEP, MA, V, XOM and CVX all
survived the sample; FOXA/FOX exists in its current form only because the 2019 Disney
transaction happened to leave two listed share classes.

Pairs that would have entered a contemporaneous 2015 universe and then disappeared —
through acquisition, de-listing or bankruptcy — never had a chance to be selected. Those
are disproportionately the cases where a spread diverges and *never* reconverges, which is
exactly the scenario that ruins a mean-reversion strategy.

**Why it is not corrected.** Doing so honestly needs a point-in-time constituent database
with de-listing returns (CRSP or equivalent), which this project does not have. Any
cheaper fix would be cosmetic.

**Why the conclusion survives anyway.** The bias runs **in favour of the strategy**. The
universe is filtered toward pairs that stayed cointegrated, and the result is still
negative. A survivorship-free test would make it worse, not better. This is the one place
where a known bias is tolerable: it is pushing against the conclusion being drawn.

The direction of a bias matters more than its size. A positive result from this universe
would be nearly uninterpretable; a negative one is safe.


### 2.3 No data-snooping — and what it would have been worth

**What it demands.** The strategy must not be selected using the data it is judged on —
neither the pairs nor the parameters.

**What was done.**

- The three pairs were fixed in advance from economic reasoning, written into
  `configs/pairs.yaml` with their rationale, and never changed.
- KO/PEP, MA/V and XOM/CVX were added later. They are tagged `sanity_check` in the config,
  reported in a separate table, and excluded from the headline. The separation is enforced
  by the code — `to_frame` carries the group label into every row — so a later-added pair
  cannot be tabulated as though it had been pre-specified.
- `window`, `entry` and `exit` were chosen a priori: 60 days ≈ a trading quarter, 2.0/0.5
  are the textbook bands.

**The uncomfortable part.** The single profitable pair in the whole study is KO/PEP, at
**+14.1% net out-of-sample** — and it is one of the ones added afterwards. If the
pre-registered set had happened to contain KO/PEP and the others had been the late
additions, the identical numbers would have supported the opposite headline. Nothing about
the data would have differed; only the order in which the pairs were written down.

**What one free parameter would have bought.** The z-score window was frozen at 60. Below,
the *only* thing that varies is that window — every other parameter, the in-sample-only
hedge fit and the cost model are untouched — and the out-of-sample net return is
recomputed for the three pre-specified pairs.


In [ ]:
GRID = [40, 50, 60, 75, 90, 120]

prices_raw = load_or_download(
    list(cfg.tickers), cfg.data.start, cfg.data.end, ROOT / cfg.data.cache_dir
)

sweep = {}
for pair in cfg.official_pairs:
    for w in GRID:
        tuned = dataclasses.replace(
            cfg, signal=dataclasses.replace(cfg.signal, window=w)
        )
        run = run_pair(pair, prices_raw, tuned)
        sweep[(pair.name, w)] = run.metrics["out_of_sample"]["net"]["total_return"] * 100

sweep_table = pd.Series(sweep).unstack().reindex(OFFICIAL)
sweep_table.columns.name = "z-score window"

print("NET out-of-sample total return %, varying ONLY the z-score window:")
display(sweep_table.round(1))
print("\nnumber of pairs showing a profit, by window:")
display((sweep_table > 0).sum(axis=0).to_frame("pairs profitable").T)


Read the bottom row. At the **pre-registered window of 60, zero of three pairs are
profitable**. At a window of **75 — an equally ordinary-looking choice — two of three
are.** WM/RSG alone spans −18.2% to +9.5% across the grid, a 27.7-point swing with a sign
change, on the same data, the same costs and the same pairs.

So a version of this project that reported "we tested windows from 40 to 120 and used 75"
could have shown a majority of its pre-specified pairs making money out-of-sample, and
every sentence in it would have been factually true. The choice of 60 is what makes the
result mean anything, and 60 was fixed before any of these numbers existed.

This is also why the sweep is reported as a **finding about parameter sensitivity** rather
than used as a parameter search. The correct conclusion to draw from the table is not "use
75" — it is that **out-of-sample results this unstable under a single innocuous knob are
not evidence of an edge in either direction.** A strategy whose sign depends on the
lookback is measuring noise.


### 2.4 Gross and net, side by side

**What it demands.** No return is reported without its after-cost counterpart.

**What was done.** `summary()` returns `{"gross": ..., "net": ...}` as a single block, so
the two cannot be separated by accident; every row of `metrics.csv` carries a `basis`
column. Reporting gross alone is not a discipline the analyst has to remember — it is not
reachable through the API.


In [ ]:
rows = []
for pair in OFFICIAL:
    for period in ["in_sample", "out_of_sample"]:
        sel = metrics[(metrics.pair == pair) & (metrics.period == period)]
        g = sel[sel.basis == "gross"].iloc[0]
        n = sel[sel.basis == "net"].iloc[0]
        rows.append({
            "pair": pair,
            "period": period,
            "gross ann %": g.annualized_return * 100,
            "net ann %": n.annualized_return * 100,
            "gross Sharpe": g.sharpe,
            "net Sharpe": n.sharpe,
        })
display(pd.DataFrame(rows).set_index(["pair", "period"]).round(2))


**The gross numbers are already unremarkable** — between −2.9% and +0.3% a year. This
matters for how the result should be read. Costs are not destroying a real edge here;
there was no edge above zero to destroy. Costs decide the *sign*, which is a weaker and
more honest claim than "costs ate the profits".

SPY/VOO is the sharpest illustration. In-sample its gross Sharpe is positive (+0.46) and
its net Sharpe is **−1.27**; out-of-sample, gross ≈ 0 becomes **−2.36** net. The two ETFs
track the same index, so the spread is extremely tight: a tiny gross edge, a tiny standard
deviation, and a cost charge that does not shrink along with them. The tightest, most
"reliable" pair in the study has the worst risk-adjusted outcome — the opposite of the
intuition that drives people toward such pairs.

`04_backtest_results.ipynb` §4 shows this drag reconciles to
`turnover × (1 + |g|) × 6bps` within 3.4%, so it is arithmetic rather than an unexplained
residual.


### 2.5 In-sample and out-of-sample, separated

**What it demands.** Everything estimated from data comes from the in-sample window; the
out-of-sample window is scored once, with everything frozen, and reported as it comes.

**What was done.** The split is 2021-12-31 — roughly 7 years in / 3 years out, a round
date chosen for its position in the sample rather than for its effect on results. The only
quantity fit from data is the static sizing hedge ratio, and it is fit on in-sample rows
only (`study.py`), then applied unchanged across the boundary.

**Why the split date itself is not a hidden parameter.** It was fixed once, and the
out-of-sample period was never re-scored under an alternative. Sweeping the split date
until the result looked better would be the same offence as sweeping the window in §2.3.


In [ ]:
view = metrics[(metrics.basis == "net") & (metrics.period != "full") & (metrics.pair.isin(OFFICIAL))]
is_oos = view.pivot_table(index="pair", columns="period", values="total_return").reindex(OFFICIAL)
display((is_oos[["in_sample", "out_of_sample"]] * 100).round(1))

print("negative cells:", int((is_oos < 0).sum().sum()), "of", is_oos.size)


**No degradation from in-sample to out-of-sample, because there was nothing to
degrade from.** The usual tell of an overfit strategy — good in-sample, collapsing
out-of-sample — is absent, since the in-sample results are already negative. Two of three
pairs are in fact *less* negative out-of-sample.

This is worth stating plainly rather than dressed up: the strategy is consistently
unprofitable, and consistency is the only positive thing that can be said about it.


## 3. What would change this conclusion

A negative result is only useful if it is clear about its own scope. Each of these could
plausibly overturn the finding, and none is tested here:

1. **Lower costs.** The drag is 0.6–1.1%/yr against a gross return near zero. An
   institution crossing the spread at a fraction of 6 bps per side changes the sign
   arithmetic — though §2.4's point stands: the gross edge would still be ~0, so this buys
   break-even, not profit.
2. **Higher frequency.** Daily bars may simply be too coarse for the horizon on which
   these spreads mean-revert. Intraday data would test that, and would also make the cost
   problem worse per unit of signal.
3. **A larger pre-registered universe.** Three pairs is a very small sample. A properly
   pre-registered set of 50–100 economically-linked pairs would distinguish "these three
   have no edge" from "this class of strategy has no edge" — a distinction this study
   cannot make.
4. **Point-in-time data.** Removing the survivorship exposure of §2.2 would make the test
   fair, and is expected to move the result further negative.
5. **Different position sizing.** Fixed unit-spread positions ignore volatility. Scaling
   by spread volatility or by conviction is a real strategy improvement and is untested
   here.

What would *not* change it: re-tuning the window, the entry/exit bands, or the split date
on this data. §2.3 shows those choices can produce whatever headline is wanted, which is
why they are frozen.


## 4. Limitations

- **Three pairs, six tickers, one decade, one parameterization.** The result is about
  *these* pairs, not about pairs trading.
- **FOXA/FOX has 709 in-sample days** against 1,763 for the others — FOX only lists from
  2019-03-13 — so its hedge ratio is fit on under half the data and all its statistics are
  noisier.
- **No statistical test on the returns themselves.** The claim is "negative", not
  "significantly negative"; with Sharpes this close to zero and samples this short, a
  formal test would fail to reject either way. Reporting it as an effect size rather than
  a p-value is the honest option.
- **Costs are a flat per-side assumption.** Real slippage varies with size, volatility and
  time of day. The flat 6 bps is a reasonable retail-taker estimate on liquid US large
  caps, not a market-impact model.
- **No borrow costs, financing, or short-availability constraints.** Every pair trade is
  half short. Adding these would push the result further negative.
- **The in-sample period contains COVID.** March 2020 dominates several pairs' spread
  behaviour; `03_backtest_explore.ipynb` shows SPY/VOO's entire gross return in an early
  full-sample run came from that window.


## 5. Conclusion

Classic pairs trading on three pre-specified, economically-linked US large-cap pairs does
not survive realistic transaction costs. All three lose money in-sample and
out-of-sample. Gross returns are approximately zero before costs, and a 0.6–1.1%/yr drag —
quantitatively explained by turnover — determines every sign.

The strategy's own logic works against it. The tighter the economic link, the more reliable
the mean reversion and the smaller the spread; but costs do not shrink with the spread.
SPY/VOO, the most tightly linked pair in the study, has the worst net Sharpe (−2.36
out-of-sample) precisely because its edge is the smallest. **The pairs most likely to
revert are the ones least able to pay for the trip.**

Two things make this a result rather than an opinion:

- The pipeline is capable of producing a positive answer. §2.3 shows a single
  ordinary-looking change to one frozen parameter turns 0-of-3 profitable into 2-of-3. The
  negative headline is a consequence of pre-registration, not of a broken backtest.
- The failure mechanism is measured, not assumed. The cost drag reconciles to
  `turnover × (1 + |g|) × 6bps` within 3.4%.

The strongest evidence in the study is the §2.3 sweep, and it cuts against strong claims in
*either* direction: an out-of-sample sign that flips on the choice of a lookback window is
not evidence of an edge, and it is not solid evidence of its absence either. It is evidence
that on this data, at this frequency, with this many pairs, the signal-to-noise ratio is
too low to support a confident claim. The defensible conclusion is the narrow one — **these
three pairs, traded this way, do not pay for their own execution** — and resisting the
temptation to state it more broadly is the same discipline as freezing the window.
